# 34 · 写第一个 Skill：从模板到 5 段完整

> **学习目标**：用 5 段模板（frontmatter / 何时用 / 何时不用 / 步骤 / 示例）从零写一个 Skill，并写一个**反例 Skill** 让自己体会「哪些 description 写错就召不回」。
>
> **预备**：33 号跑过。
>
> **为什么重要**：写坏一个 Skill 容易（description 太宽、没写反例、步骤太抽象），写好需要刻意。**写完实测召回** 是唯一标准。

In [1]:
import os, shutil
from pathlib import Path
SBX = Path('./_skill_sandbox').resolve()
if SBX.exists():
    shutil.rmtree(SBX)
SBX.mkdir()
HOME_SKILLS = SBX / 'home' / '.claude' / 'skills'
HOME_SKILLS.mkdir(parents=True)
print(f'沙箱已建: {SBX}')

沙箱已建: F:\source\code\direction\rag\learning-roadmap\03-Claude-Skills\practice\stage2_进阶\_skill_sandbox


## 1. SKILL.md 5 段完整模板（直接抄）

```
---
name: <kebab-name>            # 必须，目录名同
description: <一句话 + 何时触发 + Do NOT trigger for: X>   # **最重要**，决定召回
---

# When to use
什么场景用（1 段话）

# When NOT to use
什么场景**不要**用（反例，最容易忘）

# Steps
1. ...
2. ...

# Example
输入：...
输出：...
```

In [2]:
# 写一个真实有用的 Skill：/commit-pr
# 用途：把当前 git 改动整理成一个 PR description（标题 + 描述 + 测试清单）
skill_dir = HOME_SKILLS / 'commit-pr'
skill_dir.mkdir()
skill_md = skill_dir / 'SKILL.md'
skill_md.write_text('''---
name: commit-pr
description: When the user wants to commit current changes and open a pull request, or when the user says "commit", "create PR", "open a PR", "提 PR", "commit 一下", "提个 PR". Use this skill to generate the commit message and PR title/description from the current diff. Do NOT trigger for: just running `git status` or `git log` (use Bash directly), explaining what a commit does, or rolling back a commit.
---

# When to use
User has accumulated **unstaged or staged changes** they want to commit and push as a PR. The change set is **non-trivial** (more than just typo fixes) and they want help crafting the message.

# When NOT to use
- User just wants to check what changed → answer with `git status` + `git diff` directly
- User wants to revert / undo → use `git reset` or `git revert` directly, **do not trigger this skill**
- Single-line change → too small for a Skill, just commit inline

# Steps
1. Run `git status` to see what files changed
2. Run `git diff --staged` (and `git diff` if anything unstaged)
3. Analyze the changes:
   - Summarize the **intent** (what + why, not how)
   - Group by logical concern if multiple things changed
4. Draft a **PR title** (≤ 72 chars, imperative mood, no period at end)
5. Draft a **PR description** with these sections:
   - **What** — 1-2 sentence summary
   - **Why** — the problem or motivation
   - **How** — implementation notes (1-3 bullets)
   - **Test plan** — checkbox list, leave empty if no tests added
6. Run `git add -A` then `git commit -m "<title>"` with the title
7. Run `git push -u origin HEAD`
8. Open PR via `gh pr create --title "<title>" --body "<description>"`
9. If `gh` is not authenticated or installed, **stop and tell the user** instead of guessing the URL

# Example
User says: "commit this and open a PR"
You find 3 files changed: a new function in `utils.py`, a new test, an updated README.
PR title: `Add retry-with-jitter helper for HTTP calls`
PR description sections populated automatically.

User says: "revert my last commit"
→ Do **not** trigger this skill. Say: "If you want to undo the commit, use `git reset --soft HEAD~1` (keeps changes) or `git reset --hard HEAD~1` (discards). Want me to run either?"
''', encoding='utf-8')
print(f'已建 Skill: {skill_dir.name}/SKILL.md  ({(skill_md.stat().st_size)} bytes)')

已建 Skill: commit-pr/SKILL.md  (2279 bytes)


## 2. 写 Skill 的「好坏 description 对照」

**description 是 Skill 唯一被自动看到的字段**。坏 description = 再好的 Skill 永远不被触发。

| 写法 | 效果 |
|------|------|
| `"useful when doing things with code"` | 永远召不回（太宽） |
| `"Trigger phrases: commit, create PR, open a PR"` | 准确 |
| `Do NOT trigger for: ...` 明确 | 防止误召 |
| 例子既给正面也给了反例 | 测试用得到 |

In [3]:
# 写 3 个对照的 description，演示好坏
descriptions = {
    'BAD-太宽': 'Useful for any data analysis task.',
    'BAD-没说 Do NOT': 'Trigger when the user wants to commit code. Trigger phrases: commit, PR, push.',
    'GOOD-commit-pr': 'When the user wants to commit current changes and open a pull request, or when the user says "commit", "create PR", "open a PR", "提 PR", "commit 一下", "提个 PR". Use this skill to generate the commit message and PR title/description from the current diff. Do NOT trigger for: just running `git status` or `git log` (use Bash directly), explaining what a commit does, or rolling back a commit.',
}
for name, desc in descriptions.items():
    print(f'[{name}] {len(desc):>3} 字')
    print(f'  {desc[:100]}{"…" if len(desc) > 100 else ""}')
    print()

[BAD-太宽]  34 字
  Useful for any data analysis task.

[BAD-没说 Do NOT]  78 字
  Trigger when the user wants to commit code. Trigger phrases: commit, PR, push.

[GOOD-commit-pr] 389 字
  When the user wants to commit current changes and open a pull request, or when the user says "commit…



## 3. 反例 Skill：故意写「Too broad」，看反例怎么塌

**教学反例**比正面例子更值钱。写一个**「什么都做」的 Skill**，展示 description 太宽 + 步骤太抽象 + 无反例，**为什么是反模式**。

In [4]:
bad_dir = HOME_SKILLS / 'do-everything'
bad_dir.mkdir()
bad_skill = bad_dir / 'SKILL.md'
bad_skill.write_text('''---
name: do-everything
description: A useful helper for the user.
---

# Steps
1. Do something
2. Then do something else
3. Profit

# Example
See description.
''', encoding='utf-8')
print(f'已建反例 Skill: {bad_dir.name}/SKILL.md  ({bad_skill.stat().st_size} bytes)')
print()
print('为什么这是反模式:')
print('  ✗ description "A useful helper for the user"  → 太宽，LLM 不知道何时该用')
print('  ✗ 没有 trigger phrases → LLM 找不到匹配关键词')
print('  ✗ 没有 "Do NOT trigger for" → 任何事都可能误召')
print('  ✗ 步骤是废话（"do something"）→ Claude 不会按它行动')
print('  ✗ 例子是 "see description"  → 偷懒')

已建反例 Skill: do-everything/SKILL.md  (172 bytes)

为什么这是反模式:
  ✗ description "A useful helper for the user"  → 太宽，LLM 不知道何时该用
  ✗ 没有 trigger phrases → LLM 找不到匹配关键词
  ✗ 没有 "Do NOT trigger for" → 任何事都可能误召
  ✗ 步骤是废话（"do something"）→ Claude 不会按它行动
  ✗ 例子是 "see description"  → 偷懒


## 4. 附属资源：Skill 不止 markdown

**实际工作中** Skill 同目录可放：
- `template.md` / `template.txt` —— Prompt 模板
- `check.sh` / `check.py` —— 自动验证脚本
- `examples/` —— 多个 example
- `requirements.txt` / `package.json` —— 工具依赖

Skill 主体（SKILL.md）通过**步骤**指引 Claude Code 用工具调用它们。

In [5]:
# 给 commit-pr 加一个 PR template 资源
template = skill_dir / 'pr_template.md'
template.write_text('''# What

<1-2 sentence summary>

# Why

<the problem or motivation>

# How

- <implementation note 1>
- <implementation note 2>

# Test plan

- [ ] Unit tests added
- [ ] Manual testing done
- [ ] Edge case: <describe>
''', encoding='utf-8')
print(f'附 PR template: {template.name}  ({template.stat().st_size} bytes)')

# 在 SKILL.md 里引用它（修改 "PR description" 步骤）
skill_md_text = skill_md.read_text(encoding='utf-8')
updated = skill_md_text.replace(
    '5. Draft a **PR description** with these sections:',
    '5. Draft a **PR description** with these sections (use `./pr_template.md`):')
skill_md.write_text(updated, encoding='utf-8')
print()
print('--- 更新后的 SKILL.md 第 5 步 ---')
for line in updated.splitlines()[14:21]:
    print('  ', line)

附 PR template: pr_template.md  (236 bytes)

--- 更新后的 SKILL.md 第 5 步 ---
   1. Run `git status` to see what files changed
   2. Run `git diff --staged` (and `git diff` if anything unstaged)
   3. Analyze the changes:
      - Summarize the **intent** (what + why, not how)
      - Group by logical concern if multiple things changed
   4. Draft a **PR title** (≤ 72 chars, imperative mood, no period at end)
   5. Draft a **PR description** with these sections (use `./pr_template.md`):


## 5. Skill 设计 checklist（你写完每个 Skill 跑一遍）

In [6]:
checklist = [
    ('name 字段与目录名一致', 'kebab-case（短横线）'),
    ('description 有具体 trigger phrases（中英文都给）', '光写「useful」= 永远召不回'),
    ('description 含 Do NOT trigger 列表', '反例决定边界'),
    ('When to use 段落说明 1 个核心场景', '不是「所有场景」'),
    ('When NOT to use 段落明确反例', '常见误召列出来'),
    ('Steps 用编号 1/2/3 列清', '不要段落散文'),
    ('Example 至少 1 个正例 + 1 个反例', '反例：「不做这件事时该怎么说」'),
    ('附属资源用相对路径引用 (./xxx.md)', '可移植'),
    ('没硬编码绝对路径 / API key', 'Skill 可能被分享'),
    ('没把 Skill 当万能锤', '单一职责'),
]
print('Skill 设计 checklist (10 条):')
for i, (item, hint) in enumerate(checklist, 1):
    print(f'  [{i:2d}] {item}')
    print(f'         提示: {hint}')

Skill 设计 checklist (10 条):
  [ 1] name 字段与目录名一致
         提示: kebab-case（短横线）
  [ 2] description 有具体 trigger phrases（中英文都给）
         提示: 光写「useful」= 永远召不回
  [ 3] description 含 Do NOT trigger 列表
         提示: 反例决定边界
  [ 4] When to use 段落说明 1 个核心场景
         提示: 不是「所有场景」
  [ 5] When NOT to use 段落明确反例
         提示: 常见误召列出来
  [ 6] Steps 用编号 1/2/3 列清
         提示: 不要段落散文
  [ 7] Example 至少 1 个正例 + 1 个反例
         提示: 反例：「不做这件事时该怎么说」
  [ 8] 附属资源用相对路径引用 (./xxx.md)
         提示: 可移植
  [ 9] 没硬编码绝对路径 / API key
         提示: Skill 可能被分享
  [10] 没把 Skill 当万能锤
         提示: 单一职责


In [7]:
# 跑一遍我们的 commit-pr 看看 10 条都满足吗
def self_audit(skill_md_path):
    text = skill_md_path.read_text(encoding='utf-8')
    results = []
    results.append(('name 一致', '---\nname: ' in text and skill_md_path.parent.name in text))
    results.append(('trigger phrases', 'Trigger phrases' in text or 'When the user' in text))
    results.append(('Do NOT trigger', 'Do NOT trigger' in text))
    results.append(('When to use',  'When to use' in text))
    results.append(('When NOT to use', 'When NOT to use' in text))
    results.append(('Steps 编号',  '1.' in text and '2.' in text and '3.' in text))
    results.append(('Example 存在', 'Example' in text))
    results.append(('资源用相对路径', './' in text or '`' in text))
    results.append(('无绝对路径', 'C:\\\\' not in text and '/Users/' not in text))
    # 单一职责：description 长度 ≤ 500 字符
    desc = text.split('description:')[1].split('\n')[0] if 'description:' in text else ''
    results.append(('description ≤ 500 字（不宽）', len(desc) <= 500))
    return results

for name, dir in [('commit-pr (good)', HOME_SKILLS / 'commit-pr'), ('do-everything (bad)', HOME_SKILLS / 'do-everything')]:
    print(f'\n=== {name} ===')
    for i, (item, ok) in enumerate(self_audit(dir / 'SKILL.md'), 1):
        icon = '✅' if ok else '❌'
        print(f'  {icon} [{i:2d}] {item}')


=== commit-pr (good) ===
  ✅ [ 1] name 一致
  ✅ [ 2] trigger phrases
  ✅ [ 3] Do NOT trigger
  ✅ [ 4] When to use
  ✅ [ 5] When NOT to use
  ✅ [ 6] Steps 编号
  ✅ [ 7] Example 存在
  ✅ [ 8] 资源用相对路径
  ✅ [ 9] 无绝对路径
  ✅ [10] description ≤ 500 字（不宽）

=== do-everything (bad) ===
  ✅ [ 1] name 一致
  ❌ [ 2] trigger phrases
  ❌ [ 3] Do NOT trigger
  ❌ [ 4] When to use
  ❌ [ 5] When NOT to use
  ✅ [ 6] Steps 编号
  ✅ [ 7] Example 存在
  ❌ [ 8] 资源用相对路径
  ✅ [ 9] 无绝对路径
  ✅ [10] description ≤ 500 字（不宽）


In [8]:
shutil.rmtree(SBX, ignore_errors=True)
print('沙箱清理')

沙箱清理


## 深入思考

1. **为什么 description 写「Do NOT trigger」但 Claude 还是可能误召？**
   - description 是**软建议**，不是**硬过滤**。Claude 仍会按 prompt 里整体语境判断。要严格隔离 → **项目级 Skill 覆盖 + tools 命名克制**。
2. **Skill 多长合适？**
   - 经验：50-300 行 markdown。**太长 = LLM 抓不住重点**；**太短 = 步骤模糊**。当 Steps 超过 10 步 → 拆成多个 Skill。
3. **Skill 能调用另一个 Skill 吗？**
   - 间接：Steps 写「先 `/skill-a`，再基于结果 `/skill-b`」。**实际生产多见**。
4. **Skill 名字 `commit-pr` vs `create-pr` 哪个好？**
   - 习惯：**动作优先**（commit / build / analyze），不要名词（pr-helper）。LLM 看「create X」往往把它当生成任务；看「commit X」知道是 git 操作。
5. **多语言支持？**
   - description 双语（中英都给）能显著提升双语用户的召回。**正文中英都行**，description 是给 LLM 看的召回依据，多语言写更稳。

**改一改**：
- 把 `do-everything` 的 description 改好，看 self_audit 全部转 ✅
- 写一个属于你本机的 Skill：例如 `/open-rag-test` 跑 rag_project 的 eval

## 自检 ✅

- [ ] 默写 SKILL.md 5 段模板
- [ ] 默背 Skill 设计 checklist 10 条
- [ ] 解释「为什么 description 写宽了反而召不回」
- [ ] 解释「为什么 Steps 用编号不用散文」
- [ ] 给一个写坏的 Skill，能立刻指出 3 个改动点
- [ ] 解释「Do NOT trigger 是软建议而不是硬过滤」

## 下一步

→ [`35_description_recall_test.ipynb`](35_description_recall_test.ipynb)